## 0) Convert yolo label to json

In [ ]:
from myfile import *
import glob
from PIL import Image
import json
import os
import numpy as np

statuss = ['train', 'valid', 'test']

for status in statuss:
    image = f'C:/GroundingDINO-Med-main/floor_plans/floor_plans/{status}/images'
    label = f'C:/GroundingDINO-Med-main/floor_plans/floor_plans/{status}/labels'
    output = image + '/' + '_annotations.coco.json'

    cls_mapping = {0: 'door symbol', 1:'window symbol', 2:'zone'}
    #cls_mapping = {0: 'entrance'}

    cls_mapping = {0: 'fire', 1: 'smoke'}
    #cls_mapping = {0: 'vertebra', 1: 'scoliosis spine', 2:'normal spine'}

    json_cat = create_json_categories(cls_mapping) 
    json_img = []
    json_annotations = []
    idx,image_idx = 1, 1
    set_img = set()

    for label_path in glob.glob(label + '/' + '*.txt'):
        base_name = os.path.basename(label_path)[:-4] + '.jpg'
        img_path = os.path.join(image, base_name)
        img = Image.open(img_path)
        img_width, img_height = img.size

        json_img.append(create_json_images(set_img, base_name, img_width, img_height))

        with open(label_path, 'r') as f:
            for line in f:
                yolo = line.split(' ')

                if len(yolo) > 5:
                    cls_int = int(yolo[0])
                    polygon = list(map(float, yolo[1:]))
                    points = list(zip(polygon[0::2], polygon[1::2]))
                    xs = [x for x, y in points]
                    ys = [y for x, y in points]

                    xmin, xmax = min(xs), max(xs)
                    ymin, ymax = min(ys), max(ys)
                    
                    x_min = xmin * img_width
                    y_min = ymin * img_height
                    x_max = xmax * img_width
                    y_max = ymax * img_height

                else:
                    cls_int, cx, cy, w, h = yolo
                    cxcywh_bbox = (float(cx), float(cy), float(w), float(h))
                    x_min, y_min, x_max, y_max = cxcywh2xyxy(cxcywh_bbox, img_width, img_height)

                w = x_max - x_min
                h = y_max - y_min
                bbox = (np.floor(x_min), np.floor(y_min), np.floor(w), np.floor(h))
                json_annotations.append(create_json_annotation(int(cls_int), bbox))

    new_coco = {
                'categories' : json_cat,
                'images': json_img,
                'annotations': json_annotations
            }

    with open(output, 'w') as f:
        json.dump(new_coco, f, indent = 5)



## 1) Extract Images from the Class in json file 

In [ ]:
from myfile import ExtractClass

# status = 'valid'
# cls_name = 'ild'

#cls = ['vertebra', 'scoliosis spine', 'normal spine']
#cls = ['fire', 'smoke']
#cls = ['entrance']
cls = ['door symbol', 'window symbol', 'zone']
#cls = ['jellyfish', 'penguin', 'shark', 'starfish', 'stingray']

status = ['train', 'valid', 'test']
image = f'C:/GroundingDINO-Med-main/floor_plans/floor_plans/{status}/images'
path = 'C:/GroundingDINO-Med-main/floor_plans/floor_plans'
target_path = 'C:/GroundingDINO-Med-main/floor_plans/data_per_class/'

for c in cls:
    for s in status:
        ec = ExtractClass(path, target_path, s , c, yolo_format = True)
        ec.filter_class_from_json()


## 2) Check the annotation in the images

In [ ]:
from myfile import annotation_images

status = ['train', 'valid', 'test']
#status = ['train']

root_path = f'C:/GroundingDINO-Med-main/floor_plans/data_per_class'
save_path = f'C:/GroundingDINO-Med-main/floor_plans/data_per_class'

for c in cls:
    for s in status:
        annotation_images(root_path, save_path,c,s)

## 3) Select 5, 10, 25, 50 images per Class to demonstrate few shot learnings

In [ ]:
from myfile import Merge_Train_CLS

data_folder = 'floor_plans'
sample_size = 50

path = f'C:/GroundingDINO-Med-main/{data_folder}/sample{sample_size}/for_training'
merged_train = f'C:/GroundingDINO-Med-main/{data_folder}/sample{sample_size}/merged/train'
merged_json = f'C:/GroundingDINO-Med-main/{data_folder}/sample{sample_size}/merged/train/_annotations.coco.json'

################################# Fire ##############################################
# dict_classes = {0: 'Fire', 1: 'Smoke', }
# re_mapping = {0 : 0, 1 : 1}

################################## Aquarium ##############################################
# dict_classes = {0: 'jellyfish', 1: 'penguin', 2: 'shark', 3: 'starfish', 4: 'stingray'}
# re_mapping = {2 : 0, 3 : 1, 5: 2, 6: 3, 7: 4}

################################## Floor Plan ##############################################
dict_classes = {0: 'door symbol', 1: 'window symbol', 2: 'zone'}
re_mapping = {0 : 0, 1 : 1, 2: 2}

################################## Entrance Plan ##############################################
# dict_classes = {0: 'entrance'}
# re_mapping = {0 : 0}

################################## Vertebra ##############################################

# dict_classes = {0: 'vertebra'}
# re_mapping = {0 : 0}

Merge_Train_CLS.reset()

cls_img = f'C:/GroundingDINO-Med-main/{data_folder}/{data_folder}/train/images/'
cls_json = cls_img + '_annotations.coco.json'

mt = Merge_Train_CLS(path, cls_img, cls_json, merged_train)
mt.copy_images()
mt.save_annotations(dict_classes, re_mapping)
Merge_Train_CLS.save_json(merged_json)
count = Merge_Train_CLS.count_cls_bbox()
Merge_Train_CLS.img_lvl_cls_counting()
#Merge_Train_CLS.print_bbox(dict_classes, count)

Counter({0: 50})

In [5]:
Merge_Train_CLS.print_bbox(dict_classes, count)

For vertebra, there are 775 bbox


# 4) Randomly select x images per class for validation

In [ ]:
from myfile import Random_select

status = 'valid' # test
data_folder = 'floor_plans'
#classes = [('jellyfish', 0), ('penguin', 1), ('shark', 2), ('starfish',3), ('stingray',4)]
classes = [('door symbol', 0), ('window symbol', 1), ('zone', 2)]
#classes = [('entrance', 0)]
#classes = [('scoliosis spine', 0), ('normal spine', 1)]
#classes = [('vertebra', 0)]
#classes = [('fire', 0), ('smoke', 1)]

root_path = f'C:/GroundingDINO-Med-main/{data_folder}/data_per_class'
target_path = f'C:/GroundingDINO-Med-main/{data_folder}/sample/merged'

merged_json = f'C:/GroundingDINO-Med-main/{data_folder}/sample/merged/{status}/_annotations.coco.json'
Random_select.reset()

for cls, idx in classes:
    rdm = Random_select(root_path, target_path, cls, status)
    rdm.save_images()
    #rdm.random_save_images(75)
    rdm.save_json(idx)

Random_select.new_json(merged_json)
Random_select.count_cls()

Counter({0: 516344})

## 5) Combine all class testing together

In [ ]:
from myfile import Random_select

status = 'test' 
data_folder = 'floor_plans'
#classes = [('jellyfish', 0), ('penguin', 1), ('shark', 2), ('starfish',3), ('stingray',4)]
classes = [('door symbol', 0), ('window symbol', 1), ('zone', 2)]
#classes = [('entrance', 0)]
#classes = [('scoliosis spine', 0), ('normal spine', 1)]
#classes = [('vertebra', 0)]

root_path = f'C:/GroundingDINO-Med-main/{data_folder}/data_per_class'
target_path = f'C:/GroundingDINO-Med-main/{data_folder}/sample/merged'

merged_json = f'C:/GroundingDINO-Med-main/{data_folder}/sample/merged/{status}/_annotations.coco.json'
Random_select.reset()

for cls, idx in classes:
    rdm = Random_select(root_path, target_path, cls, status)
    rdm.save_images()
    #rdm.random_save_images(75)
    rdm.save_json(idx)

Random_select.new_json(merged_json)
Random_select.count_cls()